In [1]:
import json
import re
from pathlib import Path

RULES_PATH = Path("../docs/policy_rules.json")
policy = json.loads(RULES_PATH.read_text(encoding="utf-8"))

def _find_terms(text, terms):
    hits = []
    t_low = text.lower()
    for term in terms:
        term_low = term.lower()
        if term_low in t_low:
            hits.append(term)
    return hits

def compliance_check(text: str):
    """
    Return: {"violations": [ {rule_id, severity, matched_terms, suggestion} ... ]}
    """
    violations = []
    for rule in policy["rules"]:
        terms = []
        terms += rule.get("terms_zh", [])
        terms += rule.get("terms_en", [])
        hits = _find_terms(text, terms)
        if hits:
            violations.append({
                "rule_id": rule["id"],
                "type": rule["type"],
                "severity": rule["severity"],
                "matched_terms": hits[:20],  # 防止太长
                "suggestion_zh": rule.get("suggestion_zh", ""),
                "suggestion_en": rule.get("suggestion_en", "")
            })
    return {"violations": violations}

In [2]:
tests = [
    "这款面霜最强保湿，立刻见效，零刺激，适用于所有人。",
    "医美级配方，治疗痘痘，替代药物使用。",
    "很多人反馈质地轻薄，但敏感肌可能会刺痛，建议先做局部测试。"
]

for t in tests:
    print("TEXT:", t)
    print(compliance_check(t))
    print("-"*60)

TEXT: 这款面霜最强保湿，立刻见效，零刺激，适用于所有人。
{'violations': [{'rule_id': 'BAN_ABSOLUTE_SUPERLATIVE', 'type': 'banned_terms', 'severity': 'high', 'matched_terms': ['最强', '立刻见效', '零刺激', '适用于所有人'], 'suggestion_zh': '改为弱化表达：更适合/可能/帮助改善/多数人反馈…', 'suggestion_en': 'Use softer claims: may help / tends to / many users report...'}]}
------------------------------------------------------------
TEXT: 医美级配方，治疗痘痘，替代药物使用。
{'violations': [{'rule_id': 'RISK_MEDICAL_CLAIMS', 'type': 'risk_claims', 'severity': 'high', 'matched_terms': ['治疗', '医美级', '替代药物'], 'suggestion_zh': '删除医疗化承诺，改为：舒缓/改善观感/日常护理；必要时提示咨询专业人士', 'suggestion_en': 'Remove medical claims; use cosmetic wording; add consult-professional note if needed'}]}
------------------------------------------------------------
TEXT: 很多人反馈质地轻薄，但敏感肌可能会刺痛，建议先做局部测试。
{'violations': [{'rule_id': 'RISK_SAFETY_SENSITIVE', 'type': 'risk_claims', 'severity': 'medium', 'matched_terms': ['刺痛'], 'suggestion_zh': '加注意事项：敏感肌先局部测试/不适即停用', 'suggestion_en': 'Add caution: patch test / 